# EDA~전처리 설계 순서

## Step 0. 기본 import / 경로 설정

In [ ]:
from pathlib import Path
import re
import pandas as pd
import fitz  # PyMuPDF

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
META_PATH = DATA_DIR / "metadata" / "data_list.csv"

print("ROOT:", ROOT)
print("DATA_DIR exists:", DATA_DIR.exists())

## Step 1. 메타데이터 확인

In [ ]:
metadata = pd.read_csv(META_PATH)
metadata.head()

In [ ]:
# 컬럼 확인
metadata.columns
metadata.info()
metadata.isna().mean().sort_values(ascending=False)

## Step 2. PDF 텍스트 추출 함수

In [ ]:
def extract_pdf_text_by_page(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []

    for page_idx, page in enumerate(doc, start=1):
        text = page.get_text("text")
        pages.append({
            "page": page_idx,
            "text": text
        })

    return pages

샘플 1개 테스트:

In [ ]:
sample_pdf = next(RAW_DIR.glob("*.pdf"))
pages = extract_pdf_text_by_page(sample_pdf)

print(sample_pdf.name)
print("pages:", len(pages))
print(pages[0]["text"][:1000])

## Step 3. 텍스트 품질 EDA

In [ ]:
page_stats = pd.DataFrame([
    {
        "page": p["page"],
        "char_len": len(p["text"]),
        "line_count": len(p["text"].splitlines()),
        "empty": len(p["text"].strip()) == 0
    }
    for p in pages
])

page_stats.describe()

In [ ]:
확인할 것:
빈 페이지가 많은가?
페이지별 텍스트 길이 편차가 큰가?
표가 깨지는가?
목차/페이지 번호가 섞이는가?

## Step 4. 메타데이터 표준화 설계
문서마다 예산, 용역비용, 사업예산, 소요예산처럼 표현이 달라서 표준 필드로 바꿔야 해.

In [ ]:
FIELD_SYNONYMS = {
    "budget": ["예산", "사업예산", "용역비용", "소요예산", "사업비", "추정금액", "계약금액"],
    "period": ["사업기간", "용역기간", "계약기간", "수행기간", "과업기간"],
    "organization": ["발주기관", "수요기관", "기관명", "발주처"],
    "project_name": ["사업명", "용역명", "공고명", "과업명"],
    "contract_type": ["계약방법", "입찰방법", "낙찰자 결정방법"]
}

In [ ]:
# 텍스트에서 key-value 후보 찾기:

def find_field_candidates(text, synonyms):
    results = {}

    for field, words in synonyms.items():
        matches = []
        for word in words:
            pattern = rf"{word}\s*[:：]?\s*([^\n]+)"
            for m in re.finditer(pattern, text):
                matches.append({
                    "keyword": word,
                    "value": m.group(1).strip()
                })
        results[field] = matches

    return results

테스트:

In [ ]:
full_text = "\n".join([p["text"] for p in pages])
candidates = find_field_candidates(full_text, FIELD_SYNONYMS)
candidates

## Step 5. 섹션 분리
RFP는 Ⅰ, Ⅱ, Ⅲ, 1., 가. 같은 구조가 많아.

In [ ]:
SECTION_PATTERNS = [
    r"^[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]\.?\s+.+",
    r"^\d+\.\s+.+",
    r"^[가-하]\.\s+.+",
    r"^□\s*.+",
    r"^○\s*.+",
]

In [ ]:
섹션 후보 찾기:

In [ ]:
def detect_section_lines(pages):
    rows = []

    for p in pages:
        for line in p["text"].splitlines():
            line = line.strip()
            if not line:
                continue

            for pattern in SECTION_PATTERNS:
                if re.match(pattern, line):
                    rows.append({
                        "page": p["page"],
                        "section_line": line
                    })
                    break

    return pd.DataFrame(rows)

In [ ]:
sections_df = detect_section_lines(pages)
sections_df.head(50)

## Step 6. 요구사항 ID 추출
RFP 문서는 SFR-001, PER-001, SER-001 같은 ID가 중요함.